# Trajectory Visualization
In this section, I want to recreate the 3D trajectory visualization with welleng utilization.
hopefully I understand the welleng survey object

In [1]:
import pandas as pd 
import welleng as we


In [34]:
TrajectoryFilename = "..\\Streamlit_App\\Data\\WellTrajectory_Examples.xlsx"

TrajectoryRaw_DF = pd.read_excel(TrajectoryFilename, sheet_name="Data")
TrajectoryRaw_DF


,MD (m),Inc (°),Azi (°),TVD (m),N/S (m),E/W (m),VSEC (m),DLS (°/30m),Map N (m),Map E (m)
0,0.00,0.00,0.0,0.00,-2495.31,-4817.34,0.00,0.000,9916092.75,567043.52
1,19.20,0.00,0.0,19.20,-2495.31,-4817.34,0.00,0.000,9916092.75,567043.52
2,30.00,0.68,212.0,30.00,-2495.37,-4817.37,0.06,1.887,9916092.70,567043.49
3,60.00,2.57,212.0,59.99,-2496.09,-4817.82,0.91,1.887,9916091.98,567043.04
4,90.00,4.45,212.0,89.93,-2497.65,-4818.79,2.74,1.887,9916090.42,567042.06
...,...,...,...,...,...,...,...,...,...,...
60,1650.00,55.00,204.0,1269.23,-3343.47,-5275.88,964.16,0.000,9915244.90,566585.14
61,1680.99,55.00,204.0,1287.00,-3366.66,-5286.20,989.48,0.000,9915221.72,566574.82
62,1710.00,55.00,204.0,1303.64,-3388.37,-5295.87,1013.18,0.000,9915200.02,566565.16
63,1720.98,55.00,204.0,1309.94,-3396.59,-5299.53,1022.16,0.000,9915191.80,566561.50


In [41]:

def LoadSurvey(TrajectoryDF):
    TrajectoryDF = TrajectoryDF.copy()
    ColumnUnitDict = {}
    ColumnRenameDict = {}

    for ColName in list(TrajectoryDF.columns):
        ColKeys=ColName.split(" (")[0]
        ColValues=ColName.split(" (")[1].split(")")[0]
        ColumnRenameDict[ColName] = ColKeys
        ColumnUnitDict[ColKeys] = ColValues
    TrajectoryDF.rename(columns = ColumnRenameDict, inplace = True)
    # TrajectoryDF[]
    # TrajectoryDF[]
    # TrajectoryDF[]
    surveyObj=we.survey.Survey(TrajectoryDF['MD'].tolist(),
            TrajectoryDF['Inc'].tolist(),
            TrajectoryDF['Azi'].tolist(),
            start_xyz=[0., 0., 0.],
            deg=True,
            unit="meters",
        )


    return surveyObj
TrajectorySurvey= LoadSurvey(TrajectoryRaw_DF)

TrajectorySurvey

In [43]:
we.visual._panel(TrajectorySurvey)

In [45]:
we.visual.figure(TrajectorySurvey, type='panel')

# Manual Trajectory Design
In this section, I want to create a function that generate trajectory survey based on minimumj curvature algoritm with manually input the relative xyz coordinates.


In [65]:
import pandas as pd 
import welleng as we
def emptyTargetPoint():
    TargetDF = pd.DataFrame(columns=[
        "X",
        "Y",
        "Z",
        "DLS"
    ])
    return TargetDF
TargetDF = emptyTargetPoint()
TargetDF

,X,Y,Z,DLS


In [69]:
X_input = 300
Y_input = 25
Z_input = 1500
DLS_input = 3

TargetDF = TargetDF.append({'X': X_input, 'Y': Y_input, 'Z': Z_input, 'DLS': DLS_input}, 
                                ignore_index=True, 
                                sort=False)
TargetDF

C:\Users\irsya\AppData\Local\Temp\ipykernel_32552\1331099521.py:6: FutureWarning:

The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.



,X,Y,Z,DLS
0,0,0,0,3
1,0,0,100,3
2,300,25,800,3
3,300,25,1500,3


In [70]:
def generateSurvey(TargetDF, interpolation_step = 30, datum=15, init_inc=0, init_azi=0):
    # node_list = []
    connector_list = []
    i = 0
    for idx,row in TargetDF.iterrows():
        print([row['X'], row['Y'], row['Z'], row['DLS']])
        if i==0:

            node0 = (we.node.Node(pos=[row['X'], row['Y'], row['Z']], md=-datum, inc=init_inc, azi=init_azi))
        elif i==1:
            node = (we.node.Node(pos=[row['X'], row['Y'], row['Z']]))
            connector_list.append(
                we.connector.Connector(node0, node, dls_design=row['DLS'])
            )
            
        elif i>1:
            node = (we.node.Node(pos=[row['X'], row['Y'], row['Z']]))
            connector_list.append(
                we.connector.Connector(connector_list[-1].node_end, node,dls_design=row['DLS'])
            )
        i = i+1
    survey_example_2 = we.survey.from_connections(
        connector_list
        ).interpolate_survey(step=interpolation_step)
    return survey_example_2

In [80]:
def generateSurvey_v2(TargetDF, interpolation_step = 30, datum=15, init_inc=0, init_azi=0):
    # node_list = []
    connector_list = []
    i = 0
    for idx,row in TargetDF.iterrows():
        print([row['X'], row['Y'], row['Z'], row['DLS']])
        if i==0:
            pos1 = [row['X'], row['Y'], (row['Z'])]
            print("a")

        elif i==1:
            connector_list.append(we.connector.Connector(
                pos1=pos1,
                pos2=[row['X'], row['Y'], row['Z']],
                 md1 = -datum, inc1=init_inc, azi1=init_azi,dls_design=row['DLS']
            ))
            print("b")
        elif i>1:
            connector_list.append(we.connector.Connector(
                pos1=connector_list[-1].pos_target,
                pos2=[row['X'], row['Y'], row['Z']],
                vec1=connector_list[-1].vec_target,
                dls_design=row['DLS']

            ))
            print("c")
        i = i+1
    survey_example_2 = we.survey.from_connections(
        connector_list
        )
    return survey_example_2

In [72]:
TargetList = []
DLSList = []
for idx,row in TargetDF.iterrows():
    TargetList.append(
        [float(row['X']), float(row['Y']), float(row['Z'])]
    )
    DLSList.append(float(row['DLS']))


In [73]:
TargetList

[[0.0, 0.0, 0.0],
 [0.0, 0.0, 100.0],
 [300.0, 25.0, 800.0],
 [300.0, 25.0, 1500.0]]

In [74]:

# Push the points to the connect_points function to generate a survey
connections = we.connector.connect_points(
    TargetList,

    dls_design=3.,
    md_start=-15
    # step=30,

)
survey = we.survey.from_connections(connections)

In [75]:
we.survey.export_csv(survey, None)

,MD,INC (deg),AZI (deg),NORTHING (m),EASTING (m),TVDSS (m),DLS,TOOLFACE,BUILD RATE,TURN RATE
0,-15.000000,0.000000,0.000000,0.000000,0.000000,-0.000000,0.0,0.000000,0.000000,0.000000
1,85.000000,0.000000,0.000000,0.000000,0.000000,-100.000000,0.0,0.000000,0.000000,0.000000
2,369.977022,28.497702,4.763642,69.181917,5.765160,-373.371638,3.0,4.763642,3.000000,0.501476
3,855.423957,28.497702,4.763642,300.000000,25.000000,-800.000000,0.0,0.000000,0.000000,0.000000
4,1245.587455,10.518648,184.763642,359.586932,29.965578,-1177.968250,3.0,180.000000,1.382425,13.840352
5,1573.123265,10.518648,184.763642,300.000000,25.000000,-1500.000000,0.0,0.000000,0.000000,0.000000


In [81]:
Survey_Obj = generateSurvey_v2(TargetDF)
we.visual.figure(survey, type='scatter3d')

[0, 0, 0, 3]
a
[0, 0, 100, 3]
b
[300, 25, 800, 3]
c
[300, 25, 1500, 3]
c


In [37]:
for idx,row in TargetDF.iterrows():
    A = [row['X'], row['Y'], (row['Z'])]
A

[1500, 1500, 2500]

In [40]:
A[-1]

2500